In [49]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Set, Tuple
from collections import defaultdict
from dataclasses import dataclass, asdict
import pandas as pd
import datetime
import pickle


In [50]:
GROUND_TRUTH_PATH = Path("../data/ir_ground_truth.json")
RETRIEVAL_DIR = Path("../retrieval/laberta")
OUTPUT_DIR = Path("../evaluation")

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Model name
MODEL_NAME = "laberta_vulg"

# K values
K_VALUES = [5, 10, 20]

In [51]:
def load_ground_truth(filepath: Path) -> Tuple[Dict[str, Set[str]], Dict, Dict[str, str], Dict[str, str]]:
    print(f"Loading ground truth from: {filepath.name}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    metadata = data.get("metadata", {})
    annotations = data.get("ground_truth", [])
    
    # Build lookup dictionaries
    ground_truth_dict = {}
    query_to_type = {}
    query_to_father = {}
    
    for annotation in annotations:
        query_id = annotation["query_chunk_id"]
        relevant_chunks = set(annotation["relevant_chunks"])
        
        ground_truth_dict[query_id] = relevant_chunks
        query_to_type[query_id] = annotation.get("reference_type", "unknown")
        query_to_father[query_id] = annotation.get("church_fathers", "unknown")
    
    print(f" Loaded {len(ground_truth_dict)} annotated queries")
    print(f" Explicit: {metadata.get('by_reference_type', {}).get('explicit', 0)}")
    print(f" Implicit: {metadata.get('by_reference_type', {}).get('implicit', 0)}")
    print()
    
    return ground_truth_dict, metadata, query_to_type, query_to_father


# Load ground truth
ground_truth, gt_metadata, query_to_type, query_to_father = load_ground_truth(GROUND_TRUTH_PATH)

print(f"Total annotated queries: {len(ground_truth)}")

Loading ground truth from: ir_ground_truth.json
 Loaded 100 annotated queries
 Explicit: 50
 Implicit: 50

Total annotated queries: 100


In [52]:
# Load from pickle
retrieval_path = RETRIEVAL_DIR / f"retrieval_results_{MODEL_NAME.lower()}.pkl"

print(f"Loading from: {retrieval_path}")

with open(retrieval_path, 'rb') as f:
    retrieval_results = pickle.load(f)

print(f"  Loaded retrieval results")
print(f"  Type: {type(retrieval_results)}")
print(f"  Number of queries: {len(retrieval_results)}")

# Show structure
sample_query = list(retrieval_results.keys())[0]
sample_candidates = retrieval_results[sample_query][:3]

print(f"\nSample query: {sample_query}")
print(f"  Top 3 candidates:")
for rank, (cand_id, score) in enumerate(sample_candidates, 1):
    print(f"    {rank}. {cand_id}: {score:.4f}")

Loading from: ../retrieval/laberta/retrieval_results_laberta_vulg.pkl
  Loaded retrieval results
  Type: <class 'dict'>
  Number of queries: 19466

Sample query: 10015_sent_0
  Top 3 candidates:
    1. 066_Benedictus-Nursiae_Regula-cum-commentariis_window_520: 0.5686
    2. 066_Benedictus-Nursiae_Regula-cum-commentariis_window_698: 0.5678
    3. 031_Augustinus-Hipponensis_Confessiones_window_3: 0.5634


In [53]:
def evaluate_retrieval(
    retrieval_results: Dict[str, List[Tuple[str, float]]],
    ground_truth: Dict[str, Set[str]],
    query_to_type: Dict[str, str],
    query_to_father: Dict[str, str],  # ← Added parameter
    k_values: List[int] = [5, 10, 20]
) -> Dict:
    """
    Evaluate retrieval results with breakdown by reference type and Church Father.
    
    Args:
        retrieval_results: {query_id: [(candidate_id, score), ...]}
        ground_truth: {query_id: {relevant_candidate_ids}}
        query_to_type: {query_id: 'explicit'/'implicit'}
        query_to_father: {query_id: 'P18700' (Patrologia ID)}
        k_values: List of k values to evaluate
    
    Returns:
        Dict with overall, by_reference_type, and by_church_father results
    """
    
    # Church Father ID to name mapping
    FATHER_NAMES = {
        'P18700': 'Augustinus',
        'P17746': 'Tertullianus',
        'P18993': 'Ambrosius',
        'P18986': 'Hieronymus',
        'P18048': 'Hilarius',
        'P18988': 'Cyprian'
    }
    
    results = {}
    
    for k in k_values:
        print(f"\nEvaluating k={k}...")
        
        # Overall metrics
        precisions = []
        recalls = []
        f1_scores = []
        reciprocal_ranks = []
        
        # Metrics by reference type
        explicit_recalls = []
        explicit_mrr = []
        implicit_recalls = []
        implicit_mrr = []
        
        # Metrics by Church Father
        father_recalls = {}  # {father_id: [recall_values]}
        father_mrr = {}
        father_counts = {}
        
        queries_evaluated = 0
        queries_skipped = 0
        
        explicit_count = 0
        implicit_count = 0
        
        for query_id, relevant_set in ground_truth.items():
            ref_type = query_to_type.get(query_id, 'unknown')
            father_id = query_to_father.get(query_id, 'unknown')
            
            # Initialize father tracking
            if father_id not in father_recalls:
                father_recalls[father_id] = []
                father_mrr[father_id] = []
                father_counts[father_id] = 0
            
            # Skip if query not in retrieval results
            if query_id not in retrieval_results:
                queries_skipped += 1
                continue
            
            queries_evaluated += 1
            father_counts[father_id] += 1
            
            # Count by type
            if ref_type == 'explicit':
                explicit_count += 1
            elif ref_type == 'implicit':
                implicit_count += 1
            
            # Get top-k candidates
            retrieved = retrieval_results[query_id][:k]
            retrieved_ids = [cand_id for cand_id, score in retrieved]
            retrieved_set = set(retrieved_ids)
            
            # Compute metrics
            true_positives = len(relevant_set & retrieved_set)
            
            precision = true_positives / k if k > 0 else 0
            recall = true_positives / len(relevant_set) if len(relevant_set) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            precisions.append(precision)
            recalls.append(recall)
            f1_scores.append(f1)
            
            # MRR: Find rank of first relevant item
            rr = 0.0
            for rank, (cand_id, score) in enumerate(retrieved, 1):
                if cand_id in relevant_set:
                    rr = 1.0 / rank
                    break
            
            reciprocal_ranks.append(rr)
            
            # Store by reference type
            if ref_type == 'explicit':
                explicit_recalls.append(recall)
                explicit_mrr.append(rr)
            elif ref_type == 'implicit':
                implicit_recalls.append(recall)
                implicit_mrr.append(rr)
            
            # Store by Church Father
            father_recalls[father_id].append(recall)
            father_mrr[father_id].append(rr)
        
        # Compute averages by Church Father
        father_results = {}
        for father_id in father_recalls.keys():
            if father_recalls[father_id]:  # Has data
                father_results[father_id] = {
                    'name': FATHER_NAMES.get(father_id, father_id),
                    'recall': np.mean(father_recalls[father_id]),
                    'mrr': np.mean(father_mrr[father_id]),
                    'count': father_counts[father_id]
                }
        
        # Average metrics
        results[k] = {
            'overall': {
                'precision': np.mean(precisions) if precisions else 0.0,
                'recall': np.mean(recalls) if recalls else 0.0,
                'f1': np.mean(f1_scores) if f1_scores else 0.0,
                'mrr': np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0,
                'num_queries_evaluated': queries_evaluated,
                'num_queries_skipped': queries_skipped
            },
            'by_reference_type': {
                'explicit': {
                    'recall': np.mean(explicit_recalls) if explicit_recalls else 0.0,
                    'mrr': np.mean(explicit_mrr) if explicit_mrr else 0.0,
                    'count': explicit_count
                },
                'implicit': {
                    'recall': np.mean(implicit_recalls) if implicit_recalls else 0.0,
                    'mrr': np.mean(implicit_mrr) if implicit_mrr else 0.0,
                    'count': implicit_count
                }
            },
            'by_church_father': father_results
        }
        
        # Print overall results
        print(f"  Overall:")
        print(f"    Recall@{k}: {results[k]['overall']['recall']:.4f}")
        print(f"    MRR: {results[k]['overall']['mrr']:.4f}")
        print(f"    Queries evaluated: {queries_evaluated}/{len(ground_truth)}")
        
        # Print breakdown by reference type
        print(f"\n  By Reference Type:")
        print(f"    Explicit (n={explicit_count}):")
        print(f"      Recall@{k}: {results[k]['by_reference_type']['explicit']['recall']:.4f}")
        print(f"      MRR: {results[k]['by_reference_type']['explicit']['mrr']:.4f}")
        print(f"    Implicit (n={implicit_count}):")
        print(f"      Recall@{k}: {results[k]['by_reference_type']['implicit']['recall']:.4f}")
        print(f"      MRR: {results[k]['by_reference_type']['implicit']['mrr']:.4f}")
        
        # Print breakdown by Church Father (top 5 by count)
        print(f"\n  By Church Father (Top 5):")
        sorted_fathers = sorted(
            father_results.items(),
            key=lambda x: x[1]['count'],
            reverse=True
        )
        for father_id, metrics in sorted_fathers[:5]:
            print(f"    {metrics['name']} (n={metrics['count']}):")
            print(f"      Recall@{k}: {metrics['recall']:.4f}")
            print(f"      MRR: {metrics['mrr']:.4f}")
        
        if queries_skipped > 0:
            print(f"Queries skipped: {queries_skipped}")
    
    return results

In [54]:
print(f"Model: {MODEL_NAME}")
print(f"Ground truth entries: {len(ground_truth)}")
print(f"K values: {K_VALUES}")
print()

eval_results = evaluate_retrieval(
    retrieval_results=retrieval_results,
    ground_truth=ground_truth,
    query_to_type=query_to_type,
    query_to_father=query_to_father, 
    k_values=K_VALUES
)

Model: laberta_vulg
Ground truth entries: 100
K values: [5, 10, 20]


Evaluating k=5...
  Overall:
    Recall@5: 0.3100
    MRR: 0.2603
    Queries evaluated: 100/100

  By Reference Type:
    Explicit (n=50):
      Recall@5: 0.6000
      MRR: 0.5140
    Implicit (n=50):
      Recall@5: 0.0200
      MRR: 0.0067

  By Church Father (Top 5):
    Augustinus (n=54):
      Recall@5: 0.2593
      MRR: 0.2074
    Hieronymus (n=12):
      Recall@5: 0.3333
      MRR: 0.3333
    Tertullianus (n=9):
      Recall@5: 0.5556
      MRR: 0.4815
    Hilarius (n=5):
      Recall@5: 0.4000
      MRR: 0.4000
    Cyprian (n=4):
      Recall@5: 0.2500
      MRR: 0.1250

Evaluating k=10...
  Overall:
    Recall@10: 0.3300
    MRR: 0.2634
    Queries evaluated: 100/100

  By Reference Type:
    Explicit (n=50):
      Recall@10: 0.6400
      MRR: 0.5202
    Implicit (n=50):
      Recall@10: 0.0200
      MRR: 0.0067

  By Church Father (Top 5):
    Augustinus (n=54):
      Recall@10: 0.2593
      MRR: 0.2074
  

In [55]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# output data with breakdown
output_data = {
    "metadata": {
        "model_name": MODEL_NAME,
        "evaluation_date": str(pd.Timestamp.now()),
        "ground_truth_size": len(ground_truth),
        "k_values": K_VALUES
    },
    "results": {}
}

for k, metrics in eval_results.items():
    output_data["results"][f"k{k}"] = {
        "overall": {
            "precision": float(metrics['overall']['precision']),
            "recall": float(metrics['overall']['recall']),
            "f1": float(metrics['overall']['f1']),
            "mrr": float(metrics['overall']['mrr']),
            "num_queries_evaluated": int(metrics['overall']['num_queries_evaluated'])
        },
        "by_reference_type": {
            "explicit": {
                "recall": float(metrics['by_reference_type']['explicit']['recall']),
                "mrr": float(metrics['by_reference_type']['explicit']['mrr']),
                "count": int(metrics['by_reference_type']['explicit']['count'])
            },
            "implicit": {
                "recall": float(metrics['by_reference_type']['implicit']['recall']),
                "mrr": float(metrics['by_reference_type']['implicit']['mrr']),
                "count": int(metrics['by_reference_type']['implicit']['count'])
            }
        }
    }


# Create summary DataFrame
summary_data = []
for k in eval_results.keys():
    overall = eval_results[k]['overall']
    explicit = eval_results[k]['by_reference_type']['explicit']
    implicit = eval_results[k]['by_reference_type']['implicit']
    
    summary_data.append({
        'k': k,
        'Overall_Recall': overall['recall'],
        'Overall_MRR': overall['mrr'],
        'Explicit_Recall': explicit['recall'],
        'Explicit_MRR': explicit['mrr'],
        'Explicit_N': explicit['count'],
        'Implicit_Recall': implicit['recall'],
        'Implicit_MRR': implicit['mrr'],
        'Implicit_N': implicit['count']
    })

results_df = pd.DataFrame(summary_data)

# Save CSV
csv_path = output_dir / f"evaluation_results_{MODEL_NAME.lower()}.csv"
results_df.to_csv(csv_path, index=False)

print(f"\nResults summary:")
print(results_df.to_string(index=False))


Results summary:
 k  Overall_Recall  Overall_MRR  Explicit_Recall  Explicit_MRR  Explicit_N  Implicit_Recall  Implicit_MRR  Implicit_N
 5            0.31     0.260333             0.60      0.514000          50             0.02      0.006667          50
10            0.33     0.263429             0.64      0.520190          50             0.02      0.006667          50
20            0.35     0.265004             0.66      0.522009          50             0.04      0.008000          50
